In [ ]:
import kagglehub
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path to dataset files:", path)


In [ ]:
# Task 1: Read dataset
df = pd.read_csv(path + "/Q3_data.csv")

In [ ]:
# Task 2: Inspect first few rows
print(df.head())

In [ ]:
# Task 3: Display dataset info
print(df.info())

In [ ]:
# Task 4: Show statistical description
print(df.describe())

In [ ]:
# Task 1: Handle missing values
df = df.fillna(df.median())   # simple strategy: fill numeric NaNs with median

In [ ]:
# Task 2: Remove duplicates
df = df.drop_duplicates()

In [ ]:
# Task 3: Encode categorical variables if needed
categorical_cols = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 4: Feature scaling
scaler = StandardScaler()

num_cols = df.select_dtypes(include=['float64', 'int64']).columns

if 'target' in num_cols:
    num_cols = num_cols.drop('target')

df[num_cols] = scaler.fit_transform(df[num_cols])


In [ ]:
# Task 5: Check target imbalance
print("Target distribution:\n", y.value_counts(normalize=True))

In [ ]:
# Task 1: Split features and target
if 'target' in df.columns:
    y = df['target']
    X = df.drop('target', axis=1)
else:
    print("❌ Column 'target' not found in df. Use the correct y variable.")


In [ ]:
# Task 2: Use StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        eval_metric="F1",
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    scores.append(f1_score(y_test, y_pred))

print("Average F1 Score across folds:", np.mean(scores))


In [ ]:
# Task 1: Train model on full dataset for feature importance
full_model = CatBoostClassifier(verbose=0, random_state=42)
full_model.fit(X, y)

# Get feature importances
importances = full_model.get_feature_importance()
features = X.columns

# Sort features by importance for better visualization
sorted_idx = importances.argsort()
plt.figure(figsize=(10,6))
plt.barh(features[sorted_idx], importances[sorted_idx])
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()

In [ ]:
# Task 2: Identify golden feature
golden_feature = features[np.argmax(importances)]
print("Golden Feature:", golden_feature)

In [ ]:
X_golden = df[[golden_feature]]

scores_golden = []

for train_idx, test_idx in skf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_idx], X_golden.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(verbose=0, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    score = f1_score(y_test, y_pred)
    scores_golden.append(score)

print("Average F1 Score with Golden Feature only:", sum(scores_golden)/len(scores_golden))
